# Titanic — (Linear Regression only)

## Step 1. Load the data

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Step 2. Make features

Computers need numbers, not words. So we:
- turn **Sex** into 0/1 (male = 0, female = 1),
- fill missing **Age** with the middle value (median),
- fill missing **Fare** with the median,
- add **FamilySize** = brothers/spouses + parents/children + 1,
- add **IsAlone** = 1 if the person travels alone.

We write one small function and use it for both train and test.

In [2]:
# remember the middle values from the TRAIN data
age_median = train['Age'].median()
fare_median = train['Fare'].median()

def make_features(df):
    df = df.copy()
    # words -> numbers
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    # fill empty cells
    df['Age'] = df['Age'].fillna(age_median)
    df['Fare'] = df['Fare'].fillna(fare_median)
    # new simple features
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    return df

train = make_features(train)
test = make_features(test)

# the columns we will use
features = ['Pclass', 'Sex', 'Age', 'Fare', 'SibSp', 'Parch', 'FamilySize', 'IsAlone']

X = train[features]      # input
y = train['Survived']    # answer we want to predict
X_test = test[features]  # data for the final prediction

X.head()

,Pclass,Sex,Age,Fare,SibSp,Parch,FamilySize,IsAlone
0,3,0,22.0,7.2500,1,0,2,0
1,1,1,38.0,71.2833,1,0,2,0
2,3,1,26.0,7.9250,0,0,1,1
3,1,1,35.0,53.1000,1,0,2,0
4,3,0,35.0,8.0500,0,0,1,1


## Step 3. Train the model

In [3]:
model = LinearRegression()
model.fit(X, y)

print('Model is trained.')

Model is trained.


## Step 4. Turn the number into 0 or 1

Linear Regression gives a number like 0.2 or 0.8.
We say: if the number is **0.5 or more**, the person survived (1). If less, not (0).

First we check how good 0.5 is on our own training data.

In [4]:
# predict on the train data and compare with the real answers
train_numbers = model.predict(X)
train_guess = (train_numbers >= 0.5).astype(int)

accuracy = (train_guess == y).mean()
print('Accuracy on train data:', round(accuracy, 4))

Accuracy on train data: 0.8002


## Step 5. Make the submission file

In [5]:
# predict for the test passengers
test_numbers = model.predict(X_test)
test_guess = (test_numbers >= 0.5).astype(int)

# build the file Kaggle wants: PassengerId + Survived
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_guess
})
submission.to_csv('submission_simple.csv', index=False)

print('Saved submission_simple.csv')
submission.head()

Saved submission_simple.csv


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
